In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Crear estructura de carpetas en Drive
import os

BASE = '/content/drive/MyDrive/nla_pipeline'
carpetas = [
    f'{BASE}/checkpoints/qwen_sujeto',
    f'{BASE}/checkpoints/nla_av',
    f'{BASE}/checkpoints/nla_ar',
    f'{BASE}/activaciones',
    f'{BASE}/verbalizaciones',
    f'{BASE}/resultados',
]
for c in carpetas:
    os.makedirs(c, exist_ok=True)
    print(f'✓ {c}')

print('\n✓ Estructura de Drive lista')




Mounted at /content/drive
✓ /content/drive/MyDrive/nla_pipeline/checkpoints/qwen_sujeto
✓ /content/drive/MyDrive/nla_pipeline/checkpoints/nla_av
✓ /content/drive/MyDrive/nla_pipeline/checkpoints/nla_ar
✓ /content/drive/MyDrive/nla_pipeline/activaciones
✓ /content/drive/MyDrive/nla_pipeline/verbalizaciones
✓ /content/drive/MyDrive/nla_pipeline/resultados

✓ Estructura de Drive lista


In [2]:
# Instalar librerías necesarias
# Tiempo estimado: 3-5 minutos en primera ejecución
!pip install -q transformers>=4.40.0
!pip install -q safetensors pyyaml numpy
!pip install -q bitsandbytes>=0.43.0   # cuantización 8-bit para T4
!pip install -q accelerate             # device_map automático

# Verificar GPU disponible
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'Nombre:  {gpu.name}')
    print(f'VRAM:    {gpu.total_memory / 1e9:.1f} GB')
    vram_gb = gpu.total_memory / 1e9
    if vram_gb >= 20:
        print('→ Tienes suficiente VRAM para bfloat16 (modo Pro)')
    else:
        print('→ Usaremos cuantización 8-bit (modo Gratis/T4)')



GPU disponible: False


In [3]:
# EJECUTAR SOLO UNA VEZ — los modelos quedan guardados en Drive
from huggingface_hub import snapshot_download

BASE = '/content/drive/MyDrive/nla_pipeline'

modelos = {
    'qwen_sujeto': 'Qwen/Qwen2.5-7B-Instruct',
    'nla_av':      'kitft/nla-qwen2.5-7b-L20-av',
    'nla_ar':      'kitft/nla-qwen2.5-7b-L20-ar',
}

for carpeta, repo_id in modelos.items():
    ruta = f'{BASE}/checkpoints/{carpeta}'
    # Verificar si ya existe para no re-descargar
    if os.path.exists(f'{ruta}/config.json'):
        print(f'✓ {carpeta} ya existe — omitiendo descarga')
        continue
    print(f'Descargando {repo_id}...')
    snapshot_download(
        repo_id=repo_id,
        local_dir=ruta,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*'],
    )
    print(f'  ✓ Listo')

# Verificar que el sidecar existe (crítico para el AV)
import os
for modelo in ['nla_av', 'nla_ar']:
    sidecar = f'{BASE}/checkpoints/{modelo}/nla_meta.yaml'
    existe = '✓' if os.path.exists(sidecar) else '✗ FALTA'
    print(f'{existe}  nla_meta.yaml en {modelo}')



Descargando Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

  ✓ Listo
Descargando kitft/nla-qwen2.5-7b-L20-av...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

  ✓ Listo
Descargando kitft/nla-qwen2.5-7b-L20-ar...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

  ✓ Listo
✓  nla_meta.yaml en nla_av
✓  nla_meta.yaml en nla_ar
